In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_df = spark.table("workspace.reddit.reddit_bronze")

display(bronze_df.limit(5))

raw_json kafka_topic kafka_partition kafka_offset ingestion_timestamp {"approved_at_utc": null, "subreddit": "SwordAndSupperGame", "selftext": "This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/SwordAndSupperGame/comments/1vny7mf)", "author_fullname": "t2_4adj04u", "saved": false, "mod_reason_title": null, "gilded": 0, "clicked": false, "title": "1 or 2 \u2b50\ufe0f \ud83c\udf89 Lvl 401!!! Max Gold Drop \ud83c\udf89", "link_flair_richtext": [], "subreddit_name_prefixed": "r/SwordAndSupperGame", "hidden": false, "pwls": 6, "link_flair_css_class": null, "downs": 0, "thumbnail_height": null, "top_awarded_type": null, "hide_score": true, "name": "t3_1vny7mf", "quarantine": false, "link_flair_text_color": "dark", "upvote_ratio": 1.0, "author_flair_background_color": null, "subreddit_type": "public", "ups": 1, "total_awards_received": 0, "media_embed": {}, "thumbnail_width": null, "author_flair_template_id": null, "is_original_content": false, "user_reports": [], "secure_media": null, "is_reddit_media_domain": false, "is_meta": false, "category": null, "secure_media_embed": {}, "link_flair_text": null, "can_mod_post": false, "score": 1, "approved_by": null, "is_created_from_ads_ui": false, "author_premium": false, "thumbnail": "self", "edited": false, "author_flair_css_class": null, "author_flair_richtext": [], "gildings": {}, "content_categories": null, "is_self": true, "mod_note": null, "created": 1786684810.0, "link_flair_type": "text", "wls": 6, "removed_by_category": null, "banned_by": null, "author_flair_type": "text", "domain": "self.SwordAndSupperGame", "allow_live_comments": false, "selftext_html": "<!-- SC_OFF --><div class=\"md\"><p>This post contains content not supported on old Reddit. <a href=\"https://sh.reddit.com/r/SwordAndSupperGame/comments/1vny7mf\">Click here to view the full post</a></p>\n</div><!-- SC_ON -->", "likes": null, "suggested_sort": null, "banned_at_utc": null, "view_count": null, "archived": false, "no_follow": true, "is_crosspostable": true, "pinned": false, "over_18": false, "all_awardings": [], "awarders": [], "media_only": false, "can_gild": false, "spoiler": false, "locked": false, "author_flair_text": null, "treatment_tags": [], "visited": false, "removed_by": null, "num_reports": null, "distinguished": null, "subreddit_id": "t5_eimoap", "author_is_blocked": false, "mod_reason_by": null, "removal_reason": null, "link_flair_background_color": "", "id": "1vny7mf", "is_robot_indexable": true, "report_reasons": null, "author": "Sashimi_Rollin_", "discussion_type": null, "num_comments": 0, "send_replies": true, "contest_mode": false, "mod_reports": [], "author_patreon_flair": false, "author_flair_text_color": null, "permalink": "/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/", "stickied": false, "url": "https://www.reddit.com/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/", "subreddit_subscribers": 231064, "created_utc": 1786684810.0, "num_crossposts": 0, "media": null, "is_video": false} reddit_posts 0 1305 2026-08-14T17:11:32.765Z {"approved_at_utc": null, "subreddit": "UniversityofOtago", "selftext": "Hi,\n\nI submitted my application for halls a few days after it opened and wasn't fully aware of the fact that you had to be strategic about what halls you chose (kinda dumb ik lol but I think I got overexcited about applying) in regards to first choice and competitiveness etc, then later realised I choose 3 competitive/smaller pool ones. I read on the site that you can change your top 3 by emailing the accommodation office by September, so I am thinking of doing this for my third option. My first pick is selwyn, and my second is Knox (again, ik this is (from what I have heard) a first choice hall, but I don't want to change it as my second option because I really like it and hope that if I do not get into selwyn, I hopefully have a better chance in getting into knox than if I never put it do

In [0]:
reddit_schema = StructType([
    StructField("id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("selftext", StringType(), True),

    StructField("author", StringType(), True),
    StructField("subreddit", StringType(), True),

    StructField("score", IntegerType(), True),
    StructField("upvote_ratio", DoubleType(), True),
    StructField("num_comments", IntegerType(), True),

    StructField("created_utc", DoubleType(), True),

    StructField("url", StringType(), True),
    StructField("permalink", StringType(), True),

    StructField("over_18", BooleanType(), True),
    StructField("spoiler", BooleanType(), True)
])

In [0]:
silver_df = (
    bronze_df
    .withColumn(
        "reddit",
        from_json(col("raw_json"), reddit_schema)
    )
    .select(
        col("reddit.id").alias("post_id"),

        trim(col("reddit.title")).alias("title"),

        trim(col("reddit.selftext")).alias("selftext"),

        col("reddit.author").alias("author"),

        trim(col("reddit.subreddit")).alias("subreddit"),

        col("reddit.score").alias("score"),

        col("reddit.upvote_ratio").alias("upvote_ratio"),

        col("reddit.num_comments").alias("num_comments"),

        to_timestamp(
            from_unixtime(col("reddit.created_utc"))
        ).alias("created_utc"),

        col("reddit.url").alias("url"),

        col("reddit.permalink").alias("permalink"),

        col("reddit.over_18").alias("over_18"),

        col("reddit.spoiler").alias("spoiler"),

        # Kafka metadata
        col("kafka_topic"),
        col("kafka_partition"),
        col("kafka_offset"),
        col("ingestion_timestamp")
    )
)

In [0]:
# Remove records without a Reddit ID
silver_df = silver_df.filter(
    col("post_id").isNotNull()
)


# Create combined text for sentiment analysis
silver_df = silver_df.withColumn(
    "text",
    concat_ws(
        " ",
        coalesce(col("title"), lit("")),
        coalesce(col("selftext"), lit(""))
    )
)

In [0]:
# Remove HTML tags
silver_df = silver_df.withColumn(
    "text",
    regexp_replace(
        col("text"),
        r"<[^>]+>",
        " "
    )
)


# Remove URLs
silver_df = silver_df.withColumn(
    "text",
    regexp_replace(
        col("text"),
        r"http\S+|www\S+",
        " "
    )
)


In [0]:
# Normalize multiple spaces
silver_df = silver_df.withColumn(
    "text",
    regexp_replace(
        col("text"),
        r"\s+",
        " "
    )
)


# Remove leading and trailing spaces
silver_df = silver_df.withColumn(
    "text",
    trim(col("text"))
)

In [0]:
# Remove records where text is empty
silver_df = silver_df.filter(
    length(col("text")) > 0
)


display(silver_df.limit(10))

post_id,title,selftext,author,subreddit,score,upvote_ratio,num_comments,created_utc,url,permalink,over_18,spoiler,kafka_topic,kafka_partition,kafka_offset,ingestion_timestamp,text
1vny7mf,1 or 2 ⭐️ 🎉 Lvl 401!!! Max Gold Drop 🎉,This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/SwordAndSupperGame/comments/1vny7mf),Sashimi_Rollin_,SwordAndSupperGame,1,1.0,0,2026-08-14T05:20:10.000Z,https://www.reddit.com/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/,/r/SwordAndSupperGame/comments/1vny7mf/1_or_2_lvl_401_max_gold_drop/,false,false,reddit_posts,0,2105,2026-08-15T04:23:08.646Z,1 or 2 ⭐️ 🎉 Lvl 401!!! Max Gold Drop 🎉 This post contains content not supported on old Reddit. [Click here to view the full post](
1vny7mb,halls,"Hi, I submitted my application for halls a few days after it opened and wasn't fully aware of the fact that you had to be strategic about what halls you chose (kinda dumb ik lol but I think I got overexcited about applying) in regards to first choice and competitiveness etc, then later realised I choose 3 competitive/smaller pool ones. I read on the site that you can change your top 3 by emailing the accommodation office by September, so I am thinking of doing this for my third option. My first pick is selwyn, and my second is Knox (again, ik this is (from what I have heard) a first choice hall, but I don't want to change it as my second option because I really like it and hope that if I do not get into selwyn, I hopefully have a better chance in getting into knox than if I never put it down at all yk). My third I didn't too too much research into so I was not aware it was competitive (....carrington. im embarrassed to write this haha, I swear I had no idea it was hard to get into), so I was wondering if anyone may have suggestions for a safe but still good third choice option. obviously this will probably be one of the bigger halls (so if anyone is from any of the bigger ones any anecdotes and advice would be greatly appreciated) but Im not too sure about unicol.... might be too big for me haha. thanks",pinklemonade556,UniversityofOtago,1,1.0,0,2026-08-14T05:20:09.000Z,https://www.reddit.com/r/UniversityofOtago/comments/1vny7mb/halls/,/r/UniversityofOtago/comments/1vny7mb/halls/,false,false,reddit_posts,0,2106,2026-08-15T04:23:08.646Z,"halls Hi, I submitted my application for halls a few days after it opened and wasn't fully aware of the fact that you had to be strategic about what halls you chose (kinda dumb ik lol but I think I got overexcited about applying) in regards to first choice and competitiveness etc, then later realised I choose 3 competitive/smaller pool ones. I read on the site that you can change your top 3 by emailing the accommodation office by September, so I am thinking of doing this for my third option. My first pick is selwyn, and my second is Knox (again, ik this is (from what I have heard) a first choice hall, but I don't want to change it as my second option because I really like it and hope that if I do not get into selwyn, I hopefully have a better chance in getting into knox than if I never put it down at all yk). My third I didn't too too much research into so I was not aware it was competitive (....carrington. im embarrassed to write this haha, I swear I had no idea it was hard to get into), so I was wondering if anyone may have suggestions for a safe but still good third choice option. obviously this will probably be one of the bigger halls (so if anyone is from any of the bigger ones any anecdotes and advice would be greatly appreciated) but Im not too sure about unicol.... might be too big for me haha. thanks"
1vny7m7,"F23 If you want my services, you can come to my house. I am free tonight.",add me snap>>>> gracelirex,Briella23425,CosplayNSFW_,1,1.0,0,2026-08-14T05:20:09.000Z,https://www.reddit.com/r/CosplayNSFW_/comments/1vny7m7/f23_if_you_want_my_services_you_can_come_to_my/,/r/CosplayNSFW_/commen

In [0]:
# Remove duplicate posts within the current batch
silver_df = (
    silver_df
    .dropDuplicates(["post_id"])
)


# Create Silver table on first run
if not spark.catalog.tableExists("workspace.reddit.reddit_silver"):

    (
        silver_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("workspace.reddit.reddit_silver")
    )

    print("Silver table created successfully")

else:

    # Temporary view containing the current batch
    silver_df.createOrReplaceTempView("new_silver_data")

    # Insert only new Reddit posts
    spark.sql("""
        MERGE INTO workspace.reddit.reddit_silver AS target
        USING new_silver_data AS source
        ON target.post_id = source.post_id

        WHEN NOT MATCHED THEN
            INSERT *
    """)

    print("New Silver records inserted successfully")

✅ New Silver records inserted successfully


In [0]:
%sql
SELECT *
FROM workspace.reddit.reddit_silver
LIMIT 10;

post_id,title,selftext,author,subreddit,score,upvote_ratio,num_comments,created_utc,url,permalink,over_18,spoiler,kafka_topic,kafka_partition,kafka_offset,ingestion_timestamp,text
1vny7l0,"Built a case management tool to stop my dad's firm from losing docs and missing dates. It survived 3 months of daily use, so I'm making it public.",,HackStrix,LawStudentsIndia,1,1.0,0,2026-08-14T05:20:06.000Z,/r/IndiaLaw/comments/1vnxjsw/built_a_case_management_tool_to_stop_my_dads_firm/,/r/LawStudentsIndia/comments/1vny7l0/built_a_case_management_tool_to_stop_my_dads_firm/,false,false,reddit_posts,0,2119,2026-08-15T04:23:08.646Z,"Built a case management tool to stop my dad's firm from losing docs and missing dates. It survived 3 months of daily use, so I'm making it public."
1vodeho,At what point does bottomming feels good?,"So, I always saw myself as a pure top, I never bottom for any of my sexual partner. Maybe because I felt like it's too vulnerable a position, too gay (? I think I am internalized homophobic, even though I'm gay), or whatever. I'm writing a gay novel in which the book will be written from two perspectives of the same story. I have no problem describing or writing smuts from the POV of the top. But I have ZERO idea how a bottom feels during sex. I tried fingering myself but it was a meh. Then I asked myself, ""How can I know that I'm a pure top if I don't ever try taking dick?"" So, I bought something that looks like a dildo. It's an ash tray but with a dick attached to it, it's like a funny NSFW souvenir. I don't feel comfortable about having another guy fuck my ass yet. I stretched myself, lubed my ass, put the condom on the dick, then starting to slowly fuck myself with it, aiming for, what I believed to be, my prostate. The best adjective to describe the feeling is... Bland. I can understand why some people would love bottomming psychologically or mentally when it involves dom/sub dynamic. I have no questions about that. The dick has the same girth as mine (albeit very short because the gadget ain't exactly a dildo) which some people seem to agree that my dick is quite big. The entrace is a bit hard on the first few minutes, but then the fullness replaced the uncomfortable feeling on my hole. It burns a bit, but not too bad for me to stop everything. The fullness in my rear is a bit of a weird feeling and the pressure on my prostate is meh. I adjusted the pace, the angle, everything. But it was still meh. My dick wasn't getting hard or anything and I even tried imagining some scenarios that I might find attractive if I'm the bottom during sex. Still nothing. Thus, I pose some questions: at which point does bottomming becomes pleasant? Is there a LOT of difference between getting fucked by a guy and fucking yourself with a dildo? Or is it meh for like the first half an hour of getting fucked (or whatever the time threshold-of-bottom-enjoyability is) then it becomes pleasant? Or is it that it's not a role that will feel good physically, but rather the pleasure comes from your mental during sex which translates to various physical response in your body? Because like, my previous partners seem to enjoy it from the get go (likely because of the foreplay too) and they love it a lot. (Or maybe they are lying and I am deluded.) I want to know how to write the experience for my novel as accurately as possible. Thanks in advance.",MrMeepyy,askgaybros,1,1.0,0,2026-08-14T17:18:33.000Z,https://www.reddit.com/r/askgaybros/comments/1vodeho/at_what_point_does_bottomming_feels_good/,/r/askgaybros/comments/1vodeho/at_what_point_does_bottomming_feels_good/,true,false,reddit_posts,0,2123,2026-08-15T04:23:08.646Z,"At what point does bottomming feels good? So, I always saw myself as a pure top, I never bottom for any of my sexual partner. Maybe because I felt like it's too vulnerable a position, too gay (? I think I am internalized homophobic, even though I'm gay), or whatever. I'm writing a gay novel in which the book will be written from

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.reddit.reddit_silver;

total_records
2971


In [0]:
%sql
SELECT
    post_id,
    subreddit,
    title,
    selftext,
    text,
    score,
    num_comments,
    created_utc
FROM workspace.reddit.reddit_silver
LIMIT 10;

post_id,subreddit,title,selftext,text,score,num_comments,created_utc
1vny7l0,LawStudentsIndia,"Built a case management tool to stop my dad's firm from losing docs and missing dates. It survived 3 months of daily use, so I'm making it public.",,"Built a case management tool to stop my dad's firm from losing docs and missing dates. It survived 3 months of daily use, so I'm making it public.",1,0,2026-08-14T05:20:06.000Z
1vodeho,askgaybros,At what point does bottomming feels good?,"So, I always saw myself as a pure top, I never bottom for any of my sexual partner. Maybe because I felt like it's too vulnerable a position, too gay (? I think I am internalized homophobic, even though I'm gay), or whatever. I'm writing a gay novel in which the book will be written from two perspectives of the same story. I have no problem describing or writing smuts from the POV of the top. But I have ZERO idea how a bottom feels during sex. I tried fingering myself but it was a meh. Then I asked myself, ""How can I know that I'm a pure top if I don't ever try taking dick?"" So, I bought something that looks like a dildo. It's an ash tray but with a dick attached to it, it's like a funny NSFW souvenir. I don't feel comfortable about having another guy fuck my ass yet. I stretched myself, lubed my ass, put the condom on the dick, then starting to slowly fuck myself with it, aiming for, what I believed to be, my prostate. The best adjective to describe the feeling is... Bland. I can understand why some people would love bottomming psychologically or mentally when it involves dom/sub dynamic. I have no questions about that. The dick has the same girth as mine (albeit very short because the gadget ain't exactly a dildo) which some people seem to agree that my dick is quite big. The entrace is a bit hard on the first few minutes, but then the fullness replaced the uncomfortable feeling on my hole. It burns a bit, but not too bad for me to stop everything. The fullness in my rear is a bit of a weird feeling and the pressure on my prostate is meh. I adjusted the pace, the angle, everything. But it was still meh. My dick wasn't getting hard or anything and I even tried imagining some scenarios that I might find attractive if I'm the bottom during sex. Still nothing. Thus, I pose some questions: at which point does bottomming becomes pleasant? Is there a LOT of difference between getting fucked by a guy and fucking yourself with a dildo? Or is it meh for like the first half an hour of getting fucked (or whatever the time threshold-of-bottom-enjoyability is) then it becomes pleasant? Or is it that it's not a role that will feel good physically, but rather the pleasure comes from your mental during sex which translates to various physical response in your body? Because like, my previous partners seem to enjoy it from the get go (likely because of the foreplay too) and they love it a lot. (Or maybe they are lying and I am deluded.) I want to know how to write the experience for my novel as accurately as possible. Thanks in advance.","At what point does bottomming feels good? So, I always saw myself as a pure top, I never bottom for any of my sexual partner. Maybe because I felt like it's too vulnerable a position, too gay (? I think I am internalized homophobic, even though I'm gay), or whatever. I'm writing a gay novel in which the book will be written from two perspectives of the same story. I have no problem describing or writing smuts from the POV of the top. But I have ZERO idea how a bottom feels during sex. I tried fingering myself but it was a meh. Then I asked myself, ""How can I know that I'm a pure top if I don't ever try taking dick?"" So, I bought something that looks like a dildo. It's an ash tray but with a dick attached to it, it's like a funny NSFW souvenir. I don't feel comfortable about having another guy fuck my ass yet. I stretched myself, lubed my ass, put the condom on the dick, then starting to slowly fuck myself with it, aiming for, what